In [1]:
import pandas as pd
import numpy as np
import datetime as dt

file = r"C:\Users\Highlightning\Documents\Ciencia de Datos\Proyectos\Clustering (Customer Segmentation & Persona Profiling)\.gitignore\data\processed\clean_retail.csv"
df = pd.read_csv(file)

<h3>🎯 ¿Qué significan las métricas RFM?</h3>

* Recency (R - Recencia): ¿Hace cuántos días fue la última compra del cliente? (A menor número de días, más activo/fresco es el cliente).

* Frequency (F - Frecuencia): ¿Cuántas transacciones distintas ha realizado en el período?

* Monetary (M - Valor Monetario): ¿Cuánto dinero total ha gastado en la plataforma?

In [6]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days=1)
print(f"Fecha de corte/evaluación: {snapshot_date.strftime('%Y-%m-%d')}")

Fecha de corte/evaluación: 2011-12-10


In [9]:
rfm = (df.groupby("CustomerID").agg({
    # Recency: Días transcurridos desde la úlima compra
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    # Frequency: Cantidad de facturas/compras únicas
    "InvoiceNo": "nunique",
    # Monetary: Suma total gastada
    "Total_Invoice": "sum",
}).reset_index()
)

rfm.rename(columns={
    "InvoiceDate":  "Recency",
    "InvoiceNo":   "Frequency",
    "Total_Invoice":     "Monetary",
}, inplace=True)

print(f"\nResumen de la tabla RFM ({len(rfm):,} clientes únicos)")
print(rfm.head())


Resumen de la tabla RFM (4,324 clientes únicos)
   CustomerID  Recency  Frequency  Monetary
0     12347.0        2          7   4310.00
1     12348.0       75          4   1797.24
2     12349.0       19          1   1757.55
3     12350.0      310          1    334.40
4     12352.0       36          8   2506.04


<h3>📊 Asignación de Scores RFM (Cuartiles / Quintiles)</h3>
Para entender cómo se distribuyen tus clientes antes del clustering, asignaremos puntajes de 1 a 4 (o 1 a 5) usando cuantiles (pd.qcut)

Para "Recencia" (Recency) entre menor sea el número de días, mejor es la nota (4 es la mejor)

In [15]:
rfm["R_Score"] = pd.qcut(x=rfm["Recency"], q=4, labels=[4,3,2,1])
rfm["F_Score"] = pd.qcut(x=rfm["Frequency"].rank(method="first"), q=4, labels=[1,2,3,4])
rfm["M_Score"] = pd.qcut(x=rfm["Monetary"], q=4, labels=[1,2,3,4])

rfm["RFM_Segment"] = (rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str))
rfm["RFM_Score"] = rfm[["R_Score", "F_Score", "M_Score"]].astype(int).sum(axis=1)

In [16]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score
0,12347.0,2,7,4310.00,4,4,4,444,12
1,12348.0,75,4,1797.24,2,3,4,234,9
2,12349.0,19,1,1757.55,3,1,4,314,8
3,12350.0,310,1,334.40,1,1,2,112,4
4,12352.0,36,8,2506.04,3,4,4,344,11


In [18]:
""" Guardar reporte RFM """
rfm.to_csv(r"C:\Users\Highlightning\Documents\Ciencia de Datos\Proyectos\Clustering (Customer Segmentation & Persona Profiling)\.gitignore\data\processed\rfm_summary.csv", index=False)
print("¡Archivo 'rfm_summary.csv' guardado en data/processed/!")

¡Archivo 'rfm_summary.csv' guardado en data/processed/!
